# Experimentos avanzados — Detección de Deepfakes (Fase 5)

**TFM — Máster en Ciencia de Datos e IA (UCM)**

Tres análisis que enriquecen la sección de modelización de la memoria:

1. **Curva de aprendizaje** — cómo crece el AUC al aumentar el nº de vídeos.
2. **EfficientNet vs ResNet** — comparación de backbones en igualdad de condiciones.
3. **Rendimiento por método** — qué manipulaciones son más fáciles/difíciles de detectar.

> Requiere los rostros extraídos (Fase 1) y, para el experimento 1 y 3, los embeddings
> del pipeline principal (`data/processed/embeddings_manifest.csv`).


## Preparación del entorno en Colab

Monta Drive, clona/actualiza el repo e instala dependencias. **Edita `REPO_URL`**.

In [ ]:
import os
from pathlib import Path
try:
    import google.colab  # noqa: F401
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

if IN_COLAB:
    from google.colab import drive
    drive.mount("/content/drive")
    os.environ["TFM_WORKSPACE"] = "/content/drive/MyDrive/TFM_Deepfake"
    REPO_URL = "https://github.com/TU_USUARIO/TU_REPO.git"   # <-- EDITA ESTO
    PROJECT_ROOT = "/content/TFM_Deepfake_Detection"
    os.environ["TFM_PROJECT_ROOT"] = PROJECT_ROOT
    if not Path(PROJECT_ROOT).exists():
        !git clone {REPO_URL} {PROJECT_ROOT}
    else:
        !cd {PROJECT_ROOT} && git pull -q
    !pip install -q timm grad-cam gradio pyyaml tqdm seaborn
    !pip install -q --no-deps facenet-pytorch
    print("Entorno listo.")

## 0. Configuración

In [ ]:
import os, sys
from pathlib import Path
try:
    import google.colab  # noqa
    IN_COLAB = True
except ImportError:
    IN_COLAB = False
PROJECT_ROOT = Path(os.environ.get("TFM_PROJECT_ROOT", "/content/TFM_Deepfake_Detection")) \
    if IN_COLAB else (Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd())
sys.path.insert(0, str(PROJECT_ROOT))

import pandas as pd
import matplotlib.pyplot as plt

from src.utils.seeds import set_seed, load_config, get_device
from src.utils.paths import get_paths, ensure_dirs
from src.data.dataset import enumerate_videos, load_official_splits, assign_splits
from src.data.sequence_dataset import load_manifest

cfg = load_config(PROJECT_ROOT / "config" / "config.yaml")
set_seed(cfg["project"]["seed"])
paths = get_paths(cfg, PROJECT_ROOT); ensure_dirs(paths)
DEVICE = get_device()

# Inventario (para la comparación de backbones)
inv = enumerate_videos(paths["raw"], compression=cfg["dataset"]["compression"],
                       methods=cfg["dataset"]["manipulation_methods"])
splits_dir = paths["raw"] / "splits"
if splits_dir.exists():
    inv["split"] = assign_splits(inv, load_official_splits(splits_dir))

# Manifiesto del pipeline principal (EfficientNet)
manifest = load_manifest(paths["processed"])
print("Device:", DEVICE, "| vídeos:", len(inv), "| embeddings:", len(manifest))

## 1. Curva de aprendizaje: AUC vs nº de vídeos

Entrena con subconjuntos de entrenamiento crecientes, manteniendo test y validación
fijos. Demuestra cuánto importa el volumen de datos: el argumento que convierte tus
primeros resultados (pocos vídeos) en una conclusión explícita.

In [ ]:
from src.experiments.comparisons import learning_curve, plot_learning_curve

lc = learning_curve(manifest, cfg, DEVICE)   # sizes=None -> automático (20%..100%)
display(lc.round(3))
fig = plot_learning_curve(lc, save_path=paths["figures"] / "curva_aprendizaje.png")
plt.show()
lc.to_csv(paths["figures"] / "curva_aprendizaje.csv", index=False)

## 2. EfficientNet-B0 vs ResNet-50

Compara ambos backbones en igualdad de condiciones con el **pipeline fusionado**
(vídeo→embedding): la primera vez calcula los embeddings de ResNet directamente
desde los vídeos (MTCNN + CNN) y después quedan cacheados en
`data/processed/resnet50/`. Responde al requisito de comparar varias técnicas
justificando bondades y debilidades.

> Si ya ejecutaste `RUN_ALL` con `compare_backbone="resnet50"`, esto es casi
> instantáneo (todo está cacheado).

In [ ]:
from src.experiments.comparisons import compare_backbones

bb_table = compare_backbones(inv, paths["processed"], cfg, DEVICE,
                             backbones=[cfg["model"]["backbone"], "resnet50"])
display(bb_table)
bb_table.to_csv(paths["figures"] / "comparativa_backbones.csv")

## 3. Rendimiento por método de manipulación

Evalúa el modelo por separado sobre cada método. Suele revelar que Deepfakes es de
los más fáciles y NeuralTextures de los más difíciles: material muy valioso para la
sección de bondades y debilidades y para contextualizar el experimento cross-manipulation.

In [ ]:
from src.experiments.comparisons import train_eval_hybrid, per_method_metrics

embed_dim = int(manifest["embed_dim"].iloc[0])
from src.data.sequence_dataset import get_splits
parts = get_splits(manifest)
model, _, _ = train_eval_hybrid(parts, embed_dim, cfg, DEVICE)

method_table = per_method_metrics(manifest, model, cfg, DEVICE)
display(method_table)
method_table.to_csv(paths["figures"] / "metricas_por_metodo.csv")

# Gráfica de AUC por método
ax = method_table["auc"].plot(kind="bar", color="#457b9d", figsize=(7, 4))
ax.set_title("AUC por método de manipulación"); ax.set_ylabel("AUC"); ax.set_ylim(0, 1)
plt.xticks(rotation=20); plt.tight_layout()
plt.savefig(paths["figures"] / "auc_por_metodo.png", dpi=120, bbox_inches="tight")
plt.show()

## 4. Conclusiones

Completar con los resultados reales:

- **Curva de aprendizaje:** confirma (o no) que el AUC crece con el volumen de datos
  y permite estimar cuántos vídeos serían necesarios para saturar el rendimiento.
- **Backbones:** EfficientNet-B0 suele dar mejor relación rendimiento/parámetros;
  ResNet-50 es un referente clásico. Discute el compromiso para justificar la elección.
- **Por método:** identifica los métodos más difíciles, lo que orienta dónde reforzar
  el modelo y conecta con la generalización cross-manipulation.

Estas tres tablas/figuras van directas a la sección de modelización de la memoria.
